# Practical 10 — Reusable End-to-End Pipeline & Cryptographic Provenance

## Objective
Unify the entire log intelligence lifecycle into an automated, configuration-driven, restart-safe end-to-end processing pipeline.

**Simple meaning:** Combine all previous practicals into one robust, repeatable process that automatically ingests perimeter logs, normalizes them into canonical schema, enriches them with threat intelligence, predicts attacks with calibrated machine learning, correlates multi-event incidents, and seals the output with cryptographic SHA-256 provenance proofs.

## Academic Completion Gates (docs/future_plan.md)
1. **Configuration-driven execution:** Run from declarative configs without code modifications.
2. **Schema contracts & version checks:** Strict validation against canonical `UnifiedEvent` (UES).
3. **6-Stage Lifecycle:** Ingest -> Normalize -> Enrich -> Classify -> Correlate -> Export.
4. **Tamper-Evident Provenance:** Automatically generates SHA-256 cryptographic digests for inputs and outputs in a structured `manifest.json`.
5. **Reproducibility:** Documented, repeatable, and verified across diverse perimeter sources.


In [ ]:
import os
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pipeline import run_pipeline, verify_manifest
from src.schema.unified_event import UnifiedEvent
from src.provenance import calculate_file_sha256
print("[*] ULPF End-to-End Pipeline Engine initialized successfully.")


## Step 1 — Multi-Format Perimeter Telemetry Ingestion

Perimeter network devices produce heterogeneous, vendor-specific logs. In this step, we demonstrate the pipeline processing logs across multiple perimeter standards (Cisco ASA, Palo Alto PAN-OS, Snort IDS, Syslog).


In [ ]:
input_sample = Path(project_root) / "data" / "raw" / "cisco_asa.log"
events_output = Path(project_root) / "outputs" / "practical_10_events.jsonl"
alerts_output = Path(project_root) / "outputs" / "practical_10_alerts.jsonl"
manifest_output = Path(project_root) / "outputs" / "practical_10_manifest.json"

print(f"[*] Input Log File:  {input_sample}")
print(f"[*] Input SHA-256:   {calculate_file_sha256(input_sample)}")


## Step 2 — Executing the Complete 6-Stage Pipeline

We invoke `run_pipeline(...)` with all analytical stages active:
- **Auto-detection**: Determines parser from raw log content.
- **UES Normalization**: Maps source fields to network 5-tuple, action, and severity.
- **Enrichment**: Resolves GeoIP country, ASN, threat reputation, and MITRE technique.
- **ML Inference**: Evaluates calibrated Random Forest classifier with uncertainty abstention.
- **Temporal Correlation**: Evaluates multi-event sliding window rules.
- **Provenance**: Writes output records and calculates cryptographic manifest.


In [ ]:
summary = run_pipeline(
    input_file=input_sample,
    output_file=events_output,
    alerts_file=alerts_output,
    manifest_file=manifest_output,
    enrich=True,
    classify=True,
    correlate=True,
    validate=True,
    verbose=True,
)

print("\n--- Pipeline Execution Summary ---")
for k, v in summary.items():
    if k != "manifest":
        print(f"  {k:<20}: {v}")


## Step 3 — Inspecting Normalized & Enriched Output Records

Inspect the first normalized record to verify that all required canonical fields, contextual enrichment tags, and machine learning predictions are present.


In [ ]:
with open(events_output, "r", encoding="utf-8") as f:
    first_event = json.loads(f.readline())

print("[*] Canonical UES Event Sample:")
keys_to_show = [
    "event_uid", "format_name", "src_ip", "dst_ip", "src_port", "dst_port",
    "protocol", "event_action", "severity", "src_country", "src_asn",
    "mitre_technique_id", "mitre_technique_name", "weak_label", "label_confidence"
]
for k in keys_to_show:
    print(f"  {k:<22}: {first_event.get(k)}")


## Step 4 — Cryptographic Provenance Manifest & Tamper Verification

Data integrity is essential for digital evidence in cybersecurity. We inspect the generated `manifest.json` and verify its cryptographic audit trail.


In [ ]:
with open(manifest_output, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

print("[*] Generated Manifest Overview:")
print(f"  Schema Version : {manifest_data['manifest_schema_version']}")
print(f"  Execution ID   : {manifest_data['execution_id']}")
print(f"  Input SHA-256  : {manifest_data['input']['sha256']}")
print(f"  Output SHA-256 : {manifest_data['outputs']['canonical_events']['sha256']}")
print(f"  Events Count   : {manifest_data['outputs']['canonical_events']['events_count']}")

# Run cryptographic verification
is_valid = verify_manifest(manifest_output)
print(f"\n[INTEGRITY AUDIT] Manifest matches on-disk files: {is_valid}")
assert is_valid is True, "Provenance verification failed!"


## Step 5 — Tamper-Detection Demonstration

To demonstrate tamper detection, we simulate an adversary modifying one byte in the output file, and prove that `verify_manifest` immediately flags the discrepancy.


In [ ]:
# Read current contents
with open(events_output, "rb") as f:
    original_bytes = f.read()

# Append a tamper byte
with open(events_output, "ab") as f:
    f.write(b" ")

# Check verification
tampered_check = verify_manifest(manifest_output)
print(f"[*] Tampered file verification result: {tampered_check} (Expected: False)")
assert tampered_check is False, "Tamper was not detected!"

# Restore original bytes
with open(events_output, "wb") as f:
    f.write(original_bytes)

restored_check = verify_manifest(manifest_output)
print(f"[*] Restored file verification result: {restored_check} (Expected: True)")
assert restored_check is True, "Restored file did not pass verification!"

print("\n[SUCCESS] Practical 10 Complete — Full 10-Practical Academic Series Concluded.")
